# Khojki: frozen deployment evaluation
No training or model selection. Verify the selected model and five fresh reference lines, run regression tests and automatic-page CER, compare the historical baseline on the same fresh lines, measure CPU timing and robustness, and save unscored other-print/handwriting probes. All outputs must be saved and hash-verified before disconnecting.


In [ ]:
from pathlib import Path
import sys,os,json,time,zipfile,hashlib,subprocess,platform,datetime
BUNDLE=Path('/content/khojki-round2-evaluation.zip')
assert BUNDLE.exists(),'Upload prepared bundle via Files panel.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='41a6ee43be60d109bc2f17f9a740ea470b5ab8654182ee2994bb6ed86461ea4c'
ROOT=Path('/content/khojki-ocr')
with zipfile.ZipFile(BUNDLE) as z:z.extractall('/content')
for n,h in json.loads((ROOT/'cloud-manifest.json').read_text()).items():assert hashlib.sha256((ROOT/n).read_bytes()).hexdigest()==h,n
subprocess.run([sys.executable,'-m','pip','install','-q','scipy','opencv-python-headless','pytest','pypdf','pypdfium2'],check=True,timeout=180)
os.environ['PYTHONPATH']=str(ROOT/'src')
import torch
assert torch.cuda.is_available()
MODEL=ROOT/'models/round2-selected.pt';BASELINE=ROOT/'models/baseline-printed-khojki.pt';RES=ROOT/'results';RES.mkdir(exist_ok=True)
ANN=ROOT/'data/round2/train-dev.json';TEST=ROOT/'data/round2/test.json'
freeze=json.loads((ROOT/'data/round2/test-freeze.json').read_text())
assert hashlib.sha256(TEST.read_bytes()).hexdigest()==freeze['sha256']
selection=json.loads((ROOT/'experiments/round2-selection.json').read_text())
assert hashlib.sha256(MODEL.read_bytes()).hexdigest()==selection['checkpoint_sha256']
def run(args,log,timeout=300):
 with log.open('w') as f:p=subprocess.run([sys.executable,*map(str,args)],stdout=f,stderr=subprocess.STDOUT,timeout=timeout)
 print(log.name,log.read_text()[-2200:]);p.check_returncode()
(RES/'round2-environment.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'platform':platform.platform()},indent=2))
(RES/'round2-requirements.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Frozen model and fresh references verified. No training or selection in this notebook.')


In [ ]:
run(['-m','pytest',ROOT/'tests','-q'],RES/'round2-tests.log',120)
for tag,model,ann,split in [('round2-dev',MODEL,ANN,'dev'),('round2-test',MODEL,TEST,'test'),('round2-baseline-test',BASELINE,TEST,'test')]:
 if not (RES/tag/'evaluation.json').exists():
  run([ROOT/'scripts/evaluate_pages.py','--checkpoint',model,'--annotations',ann,'--device','cuda','--split',split,'--output',RES/tag],RES/(tag+'.log'))
run([ROOT/'scripts/evaluate.py','--checkpoint',MODEL,'--annotations',ANN,'--split','train','--device','cuda','--output',RES/'round2-train.json'],RES/'round2-train.log')
run([ROOT/'scripts/stress_eval.py','--checkpoint',MODEL,'--annotations',ANN,'--device','cuda','--output',RES/'round2-stress.json'],RES/'round2-stress.log')
run(['-m','khojki_ocr',ROOT/'data/pages/p19.png','--checkpoint',MODEL,'--device','cpu','--output',RES/'round2-cpu-demo','--threads','2'],RES/'round2-cpu.log')
for page,name in [(4,'round2-unscored-handwriting'),(23,'round2-unscored-other-print')]:
 run(['-m','khojki_ocr',ROOT/f'data/pages/p{page:02d}.png','--checkpoint',MODEL,'--device','cuda','--output',RES/name],RES/(name+'.log'))
cpu=(RES/'round2-cpu-demo/page-001/transcription.txt').read_text();gpu=(RES/'round2-test/p19/transcription.txt').read_text()
run([ROOT/'scripts/report_round2.py'],RES/'round2-report.log')
complete={'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'model_sha256':hashlib.sha256(MODEL.read_bytes()).hexdigest(),'test_reference_sha256':freeze['sha256'],'test_summary':json.loads((RES/'round2-test/evaluation.json').read_text())['summary'],'cpu_gpu_page19_text_equal':cpu==gpu,'cpu_page':json.loads((RES/'round2-cpu-demo/run.json').read_text()),'test_used_for_selection':False}
(RES/'round2-evaluation-complete.json').write_text(json.dumps(complete,indent=2))
print('EVALUATION_COMPLETE',json.dumps(complete,indent=2))

PYPROJECT='[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "khojki-ocr-takehome"\nversion = "0.1.0"\ndescription = "Auditable printed Khojki OCR prototype"\nrequires-python = ">=3.10"\ndependencies = ["torch>=2.2", "numpy>=1.26", "Pillow>=10", "opencv-python-headless>=4.9", "scipy>=1.12", "pypdf>=4", "pypdfium2>=4.30"]\n\n[project.optional-dependencies]\ntrain = ["uharfbuzz>=0.39", "freetype-py>=2.4"]\ntest = ["pytest>=8"]\nbaseline = ["requests>=2.31"]\n\n[project.scripts]\nkhojki-ocr = "khojki_ocr.__main__:main"\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'
# Verify the documented editable install and console entry point.
(ROOT/'pyproject.toml').write_text(PYPROJECT)
run(['-m','pip','install','-q','--no-deps','-e',ROOT],RES/'round2-package-install.log',120)
run(['-c','import subprocess; subprocess.run(["khojki-ocr","--help"],check=True)'],RES/'round2-console-help.log',30)
print('Editable install and console entry point verified.')


In [ ]:
from IPython.display import display,HTML
import base64
TARGET=Path('/content/khojki-round2-evaluation-results.zip')
with zipfile.ZipFile(TARGET,'w',zipfile.ZIP_DEFLATED) as z:
 for p in RES.rglob('*'):
  if p.is_file():z.write(p,str(p.relative_to(ROOT)))
EXPORTS=[{'name':TARGET.name,'bytes':TARGET.stat().st_size,'sha256':hashlib.sha256(TARGET.read_bytes()).hexdigest()}]
display(HTML('<a download="'+TARGET.name+'" href="data:application/zip;base64,'+base64.b64encode(TARGET.read_bytes()).decode()+'">Download verified evaluation results</a>'))
print('EXPORT_MANIFEST',json.dumps(EXPORTS))
